# 中文社交媒体文本预处理

## 1. 课程背景

你有没有想过，当我们在微博、抖音、淘宝上发评论时，这些文字是怎么被电脑分析的？

比如：
- 淘宝怎么知道这条评论是好评还是差评？
- 微博怎么统计热门话题？
- 抖音怎么推荐你感兴趣的内容？

答案是：电脑需要先"清洗"这些文字，把乱七八糟的内容整理干净，才能进行分析。

这节课，我们就来学习如何清洗社交媒体文本。

## 2. 学习目标

学完本节课，你应该能够：
- ✅ 理解为什么需要清洗社交媒体文本
- ✅ 掌握7种常见的文本清洗方法
- ✅ 理解什么是Pipeline（流水线）
- ✅ 能够处理真实的微博评论数据
- ✅ 为情感分析做好数据准备

## 3. 社交媒体文本的特点



### 3.1 什么是社交媒体文本？

社交媒体文本就是我们在网上发的各种内容：
- 微博评论
- 淘宝商品评价
- 抖音视频评论
- B站弹幕
- 小红书笔记

### 3.2 为什么需要清洗？

看看这条真实的评论：

```
@小明 你说得对！！！这个产品真的很棒😊😊 #好物推荐# http://xxx.com
```

如果直接分析，会遇到什么问题？
- `@小明` - 这是在@别人，不是评价内容
- `！！！` - 重复的标点，只需要保留一个
- `😊😊` - 表情符号，电脑不认识
- `#好物推荐#` - 话题标签，不是评价内容
- `http://xxx.com` - 网址链接，对分析没用

**清洗后的结果：**
```
你说得对！这个产品真的很棒
```

干净多了！这样电脑才能准确分析出这是一条"好评"。

### 3.3 社交媒体文本的7大特点

| 序号 | 特点 | 例子 | 为什么要去除 |
|------|------|------|--------------|
| 1 | 表情符号 | 😊❤️👍🔥 | 电脑难以理解，影响分析 |
| 2 | @用户 | @张三 @客服 | 不是评价内容，是互动信息 |
| 3 | 话题标签 | #双十一# #好物推荐# | 是分类标签，不是评价内容 |
| 4 | URL链接 | http://xxx.com | 对文本分析没有帮助 |
| 5 | 重复标点 | ！！！？？？。。。 | 重复无意义，保留一个即可 |
| 6 | 特殊符号 | 【】《》⚡ | 装饰性符号，影响分词 |
| 7 | 多余空格 | "   很  好   " | 格式不规范，需要统一 |

### 3.4 预处理的目标

把"脏数据"变成"干净数据"，方便后续的：
- 情感分析（判断好评/差评）
- 关键词提取（找出重点词汇）
- 文本分类（自动分类评论）
- 数据统计（词频分析）


## 4. 准备数据集

我们准备了20条真实场景的评论，涵盖各种情况：

In [ ]:
# 模拟微博评论数据（扩展版）
# 这些评论包含了社交媒体文本的各种典型特征
comments = [
    "@小明 你说得对！！！这个产品真的很棒😊😊 #好物推荐# http://xxx.com",
    "   质量不错。。。但是价格有点贵   ",
    "👍👍👍强烈推荐！！！@李四 你也来看看",
    "【官方旗舰店】值得购买，物流很快⚡⚡",
    "哈哈哈哈哈😂😂太搞笑了吧！！！@王五 快来看 #搞笑视频# https://weibo.com/123",
    "服务态度超级好💯💯💯，下次还会再来的~~~",
    "@客服 什么时候发货啊？？？已经等了三天了。。。😤",
    "性价比很高👏推荐给大家 #双十一# #剁手节# ",
    "   emmm...感觉一般般吧   不如上一代产品   ",
    "【限时优惠】现在下单立减50元！！！🔥🔥🔥 http://shop.com/sale",
    "@张三 @李四 @王五 姐妹们冲啊💪💪 #团购# ",
    "质量太差了😡😡😡 完全不值这个价！！！差评❌❌❌",
    "包装很精美🎁，送人很有面子，好评⭐⭐⭐⭐⭐",
    "哇塞！！！颜值爆表😍😍 #种草# #美妆# http://beauty.com",
    "   客服回复太慢了。。。   等了半小时都没人理   ",
    "【新品上市】全网最低价🎉🎉 @所有人 不要错过哦~~~",
    "用了一周，效果还不错👌 就是物流有点慢🚚",
    "这是什么神仙产品啊！！！爱了爱了❤️❤️❤️ #安利# ",
    "@官方客服 能不能退货？收到的商品有瑕疵😓",
    "性能强劲💪 外观时尚✨ 价格实惠💰 三个愿望一次满足！"
]

# 先看看原始数据长什么样
print("📊 原始评论数据（共20条）：")
print("="*70)
for i, comment in enumerate(comments, 1):
    print(f"{i}. {comment}\n")

## 5. 文本预处理步骤

### 5.1 步骤1：导入工具库

在开始清洗之前，我们需要导入一些工具：

In [ ]:
import re          # 正则表达式库，用于文本匹配和替换
import string      # 字符串处理库
import jieba       # 中文分词库
import jieba.posseg as pseg  # 词性标注库

# 库的作用说明：
# re - 用来查找和替换文本中的特定模式（如URL、表情等）
# jieba - 把句子切分成一个个词语
# pseg - 标注每个词的词性（名词、动词、形容词等）

print("✅ 工具库导入完成！")

### 5.2 步骤2：定义清洗函数

#### 5.2.1 什么是函数？

函数就像一个"加工机器"：
- **输入**：脏乱的文本
- **处理**：执行7个清洗步骤
- **输出**：干净的文本

#### 5.2.2 为什么要用函数？

如果有100条评论，难道要手动清洗100次吗？

**用函数的好处：**
- 写一次代码，重复使用
- 统一处理标准
- 方便修改和维护

#### 5.2.3 清洗函数详解

In [ ]:
def clean_social_text(text):
    """
    社交媒体文本清洗函数
    
    功能：去除社交媒体文本中的噪声信息
    输入：原始文本（字符串）
    输出：清洗后的文本（字符串）
    """
    
    # 步骤1：去除URL链接
    text = re.sub(r'http[s]?://\S+', '', text)
    
    # 步骤2：去除@用户
    text = re.sub(r'@\w+', '', text)
    
    # 步骤3：去除话题标签
    text = re.sub(r'#\w+#', '', text)
    
    # 步骤4：去除表情符号
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U00002600-\U000027BF"
        "]+", flags=re.UNICODE)
    text = emoji_pattern.sub('', text)
    
    # 步骤5：去除特殊符号
    text = re.sub(r'[【】《》⚡🎁⭐❌💯👏💪🔥🚚✨💰]', '', text)
    
    # 步骤6：规范化重复标点
    text = re.sub(r'([！。？，~])+', r'\1', text)
    
    # 步骤7：去除多余空格
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    
    return text

print("✅ 清洗函数定义完成！")

### 5.3 步骤3：测试清洗效果

我们用前5条评论来测试一下清洗效果：

In [ ]:
print("🔍 清洗效果对比：\n" + "="*70)

for i, comment in enumerate(comments[:5], 1):
    cleaned = clean_social_text(comment)
    print(f"\n【评论 {i}】")
    print(f"原文：{comment}")
    print(f"清洗后：{cleaned}")
    print(f"变化：去除了 {len(comment) - len(cleaned)} 个字符")

### 5.4 步骤4：分词处理

#### 5.4.1 什么是分词？

分词就是把一句话切分成一个个词语。

**例如：**
- 原句：`这个产品真的很棒`
- 分词后：`这个 / 产品 / 真的 / 很棒`

#### 5.4.2 为什么要分词？

中文不像英文，词与词之间没有空格。电脑需要知道哪些字组成一个词，才能理解句子的意思。


#### 5.4.3 分词演示

In [ ]:
print("📝 分词结果展示：\n" + "="*70)

for i, comment in enumerate(comments[:3], 1):
    cleaned = clean_social_text(comment)
    words = jieba.lcut(cleaned)
    print(f"\n【评论 {i}】")
    print(f"清洗后：{cleaned}")
    print(f"分词后：{' / '.join(words)}")
    print(f"词语数量：{len(words)} 个")

### 5.5 步骤5：去除停用词

#### 5.5.1 什么是停用词？

停用词就是那些出现频率很高，但对文本分析没什么帮助的词。

**常见的停用词：**
- 的、了、在、是、我、有、和、就、不、都、一、很、到、这

**例如：**
- 原句：`这个产品真的很棒`
- 去停用词后：`产品 / 真的 / 很棒`（去掉了"这个"）

#### 5.5.2 为什么要去停用词？

- 减少无用信息
- 突出关键词
- 提高分析效率

#### 5.5.3 去停用词演示

In [ ]:
stopwords = {'的', '了', '在', '是', '我', '有', '和', '就', '不', '都', '一', '很', '到', '这', '但是', '有点', '你', '也', '来', '看看'}

print("🚫 去停用词效果：\n" + "="*70)

for i, comment in enumerate(comments[:3], 1):
    cleaned = clean_social_text(comment)
    words = jieba.lcut(cleaned)
    filtered = [w for w in words if w not in stopwords and len(w) > 1]
    print(f"\n【评论 {i}】")
    print(f"分词后：{' / '.join(words)}")
    print(f"去停用词：{' / '.join(filtered)}")
    print(f"减少了 {len(words) - len(filtered)} 个词")

### 5.6 步骤6：提取关键词

#### 5.6.1 什么是关键词？

关键词就是最能代表文本主题的词语，通常是：
- **名词**：产品、质量、价格、物流、服务
- **形容词**：很棒、不错、便宜、快速


#### 5.6.2 什么是词性标注？

词性标注就是给每个词打上"身份标签"：
- **n** - 名词（noun）：产品、手机、质量
- **a** - 形容词（adjective）：很棒、不错、便宜
- **v** - 动词（verb）：购买、推荐、使用
- **d** - 副词（adverb）：很、非常、特别

#### 5.6.3 为什么只提取名词和形容词？

在商品评论中：
- **名词**告诉我们"评价什么"（产品、质量、物流）
- **形容词**告诉我们"评价如何"（很棒、不错、太差）

#### 5.6.4 关键词提取演示

In [ ]:
print("🔑 关键词提取效果：\n" + "="*70)

for i, comment in enumerate(comments[:3], 1):
    cleaned = clean_social_text(comment)
    words_pos = pseg.cut(cleaned)
    keywords = [word for word, flag in words_pos if flag.startswith('n') or flag.startswith('a')]
    print(f"\n【评论 {i}】")
    print(f"原文：{comment}")
    print(f"关键词：{' / '.join(keywords)}")
    print(f"说明：这些词最能代表评论的核心内容")

## 6. 构建完整的Pipeline



### 6.1 什么是Pipeline（流水线）？

Pipeline就像工厂的生产流水线：

```
原始文本 → 清洗 → 分词 → 去停用词 → 提取关键词 → 最终结果
```

### 6.2 类比理解

想象一个苹果加工厂：
- **原料**：带泥的苹果（脏数据）
- **步骤1**：清洗（去除泥土）
- **步骤2**：切片（分词）
- **步骤3**：去核（去停用词）
- **步骤4**：挑选（提取关键词）
- **成品**：干净的苹果片（可用数据）

### 6.3 Pipeline的优势

1. **自动化**：一次性完成所有步骤
2. **标准化**：每条数据都经过相同处理
3. **可配置**：可以选择需要的处理步骤
4. **可复用**：写一次，用无数次

### 6.4 定义Pipeline函数

In [ ]:
def preprocess_social_text(text, remove_stopwords=True, extract_keywords=False):
    """
    社交媒体文本预处理Pipeline（流水线）
    
    参数说明：
        text: 原始文本（字符串）
        remove_stopwords: 是否去除停用词（True/False）
        extract_keywords: 是否只提取关键词（True/False）
    
    返回值：
        处理后的词语列表
    """
    
    # 步骤1：文本清洗
    text = clean_social_text(text)
    
    # 步骤2：分词处理
    if extract_keywords:
        words_pos = pseg.cut(text)
        words = [w for w, f in words_pos if f.startswith('n') or f.startswith('a')]
    else:
        words = jieba.lcut(text)
    
    # 步骤3：去停用词
    if remove_stopwords:
        stopwords = {'的', '了', '在', '是', '我', '有', '和', '就', '不', '都', '一', '很', '到', '这', '但是', '有点', '你', '也', '来', '看看', '说'}
        words = [w for w in words if w not in stopwords and len(w) > 1]
    
    return words

print("✅ Pipeline函数定义完成！")

## 7. 测试Pipeline



### 7.1 Pipeline的三种使用模式

我们的Pipeline支持三种处理模式，适应不同的需求：

| 模式 | 参数设置 | 适用场景 |
|------|----------|----------|
| 模式1：基础分词 | remove_stopwords=False, extract_keywords=False | 需要保留所有词语 |
| 模式2：去停用词 | remove_stopwords=True, extract_keywords=False | 一般文本分析 |
| 模式3：提取关键词 | remove_stopwords=True, extract_keywords=True | 情感分析、主题提取 |



### 7.2 实际测试

测试文本

In [ ]:
test_comment = "@小明 这个手机拍照效果真的很棒😊😊！！！强烈推荐👍 #数码产品# http://xxx.com"

print(f"测试文本：{test_comment}\n")

模式1：基础分词

In [ ]:
result1 = preprocess_social_text(test_comment, remove_stopwords=False, extract_keywords=False)
print(f"\n【模式1：基础分词】")
print(f"结果：{' / '.join(result1)}")
print(f"说明：保留了所有词语")

模式2：去停用词

In [ ]:
result2 = preprocess_social_text(test_comment, remove_stopwords=True, extract_keywords=False)
print(f"\n【模式2：去停用词】")
print(f"结果：{' / '.join(result2)}")
print(f"说明：去除了停用词，保留有意义的词")

模式3：提取关键词

In [ ]:
result3 = preprocess_social_text(test_comment, remove_stopwords=True, extract_keywords=True)
print(f"\n【模式3：提取关键词】")
print(f"结果：{' / '.join(result3)}")
print(f"说明：只保留名词和形容词")

**选择建议**：
- 做情感分析 → 用模式3
- 做词频统计 → 用模式2
- 做文本生成 → 用模式1


## 8. 批量处理


### 8.1 什么是批量处理？

批量处理就是用Pipeline一次性处理多条评论，就像：
- 一次洗一件衣服 ❌ 效率低
- 一次洗一筐衣服 ✅ 效率高

### 8.2 批量处理演示

我们用Pipeline处理所有20条评论：

In [ ]:
print("🔄 批量处理结果（提取关键词模式）：\n" + "="*70)

for i, comment in enumerate(comments, 1):
    result = preprocess_social_text(comment, remove_stopwords=True, extract_keywords=True)
    print(f"\n【评论 {i}】{comment[:30]}...")
    if result:
        print(f"关键词：{' / '.join(result)}")
    else:
        print(f"关键词：（无名词或形容词）")

print("\n" + "="*70)
print("✅ 成功处理了 20 条评论！")


## 9. 课程总结

### 9.1 本节课学到了什么？

#### 9.1.1 社交媒体文本的7大特点

| 序号 | 特点 | 处理方法 | 正则表达式 |
|------|------|----------|------------|
| 1 | URL链接 | 删除 | `r'http[s]?://\S+'` |
| 2 | @用户 | 删除 | `r'@\w+'` |
| 3 | 话题标签 | 删除 | `r'#\w+#'` |
| 4 | 表情符号 | 删除 | Unicode范围匹配 |
| 5 | 特殊符号 | 删除 | `r'[【】《》⚡]'` |
| 6 | 重复标点 | 规范化 | `r'([！。？，])+' → r'\1'` |
| 7 | 多余空格 | 统一 | `r'\s+' → ' '` |

#### 9.1.2 文本预处理的完整流程

```
原始文本
   ↓
步骤1：文本清洗（去除噪声）
   ↓
步骤2：分词处理（切分词语）
   ↓
步骤3：去停用词（过滤无用词）
   ↓
步骤4：词性标注（标注词性）
   ↓
步骤5：关键词提取（提取名词和形容词）
   ↓
最终结果
```

#### 9.1.3 Pipeline（流水线）的概念

- **定义**：把多个处理步骤封装成一个函数
- **优势**：自动化、标准化、可配置、可复用
- **类比**：工厂流水线，原料进去，成品出来

#### 9.1.4 三种处理模式

| 模式 | 配置 | 适用场景 |
|------|------|----------|
| 模式1 | remove_stopwords=False, extract_keywords=False | 文本生成 |
| 模式2 | remove_stopwords=True, extract_keywords=False | 词频统计 |
| 模式3 | remove_stopwords=True, extract_keywords=True | 情感分析 |

### 9.2 核心函数回顾

#### 9.2.1 clean_social_text(text)
- **功能**：清洗社交媒体文本
- **输入**：原始文本
- **输出**：清洗后的文本

#### 9.2.2 preprocess_social_text(text, remove_stopwords, extract_keywords)
- **功能**：完整的预处理Pipeline
- **输入**：原始文本 + 配置参数
- **输出**：处理后的词语列表

### 9.3 实际应用场景

#### 9.3.1 电商评论分析
- 提取商品评价的关键词
- 统计用户关注的产品特性
- 为情感分析做数据准备

#### 9.3.2 社交媒体监控
- 清洗微博、抖音评论
- 提取热门话题关键词
- 分析用户情感倾向

#### 9.3.3 客服数据分析
- 处理客户反馈
- 提取常见问题
- 优化产品和服务

### 9.4 练习建议

#### 9.4.1 初级练习
- 收集10条真实的淘宝评论
- 使用Pipeline处理这些评论
- 对比处理前后的差异

#### 9.4.2 中级练习
- 修改stopwords，添加更多停用词
- 调整清洗函数，处理更多特殊符号
- 尝试不同的处理模式，观察效果

#### 9.4.3 高级练习
- 收集100条微博评论
- 使用Pipeline批量处理
- 统计词频，制作词云图
- 分析用户情感倾向（正面/负面）

### 9.5 下一步学习

掌握了文本预处理后，可以继续学习：
- 情感分析（判断好评/差评）
- 文本分类（自动分类文本）
- 关键词提取（TF-IDF、TextRank）
- 词向量（Word2Vec）
- 深度学习NLP（BERT、GPT）

### 9.6 重要提醒

- 不同的任务需要不同的预处理方式
- 没有"最好"的方法，只有"最合适"的方法
- 多实践、多尝试、多总结
- 预处理质量直接影响后续分析效果

